# Agent 1 — หน้าแชตจำลอง LINE

หน้าจอคล้าย LINE สำหรับคุยกับ Agent 1 ตัวจริง (`Agent1.handle()` + Gemini) ไม่ต้องรัน API server และไม่ส่งข้อความเข้า LINE จริง

- ข้อความและปุ่ม Quick Reply สร้างด้วย `build_messages()` ตัวเดียวกับที่ส่งให้ LINE: ข้อความยาวแยกเป็นหลาย bubble ตามจริง
- กดปุ่ม Quick Reply ได้ (ยินยอม / ยืนยัน / แก้ไข / เลือกเคส …) และกด 📍 เพื่อส่งตำแหน่ง
- แผงด้านขวาแสดงสถานะเคส: intake_status, slot ที่ขาด, Priority, Risk facts
- **↺ ผู้ใช้ใหม่** = คุยในฐานะผู้ใช้อีกคนใน DB เดียวกัน, พิมพ์ user_id เดิมเพื่อย้อนกลับไปแชตเดิม, **ล้าง DB ทั้งหมด** = เริ่มใหม่หมด

ข้อมูลเก็บใน SQLite ในหน่วยความจำ ใช้ข้อมูลสมมติเท่านั้น การเรียก Gemini อาจมีค่าใช้จ่ายตามบัญชี

1. รันเซลล์ติดตั้ง (ครั้งแรก) แล้ว restart kernel
2. ใส่ `GEMINI_API_KEY` ใน `lab/1/.env` หรือ `dspy-agent/.env`
3. รันเซลล์ตั้งค่า แล้วรันเซลล์หน้าแชต

In [ ]:
# รันครั้งแรก หากเพิ่งติดตั้งให้ restart kernel แล้วรันเซลล์ถัดไป
from pathlib import Path

BASE = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "dspy-agent" / "app" / "agent.py").is_file()), None)
if BASE is None:
    raise RuntimeError("กรุณาเปิด notebook ภายในโฟลเดอร์ DSI490")
%pip install -r {str(BASE / "dspy-agent" / "requirements-notebook.txt")}

## ตั้งค่า Gemini และ Agent

In [ ]:
import os
import sys
from pathlib import Path

BASE = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "dspy-agent" / "app" / "agent.py").is_file()), None)
if BASE is None:
    raise RuntimeError("กรุณาเปิด notebook ภายในโฟลเดอร์ DSI490")
ROOT = BASE / "dspy-agent"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import dspy
from dotenv import load_dotenv

for env_path in (BASE / "lab" / "1" / ".env", ROOT / ".env"):
    load_dotenv(env_path, override=False)
api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
if not api_key:
    raise RuntimeError("ยังไม่มี API key: เพิ่ม GEMINI_API_KEY ใน lab/1/.env แล้วรันเซลล์นี้ใหม่")
MODEL = os.getenv("GEMINI_MODEL", "gemini/gemini-3.1-flash-lite")
dspy.configure(lm=dspy.LM(model=MODEL, api_key=api_key))

from app.agent import Agent1
from app.knowledge import KnowledgeBase
from app.store import Store

agent = None


def reset_agent():
    """ล้างทุกเคส/ทุกผู้ใช้ เริ่ม SQLite ในหน่วยความจำใหม่"""
    global agent
    if agent is not None:
        agent.store.close()
    agent = Agent1(Store("sqlite:///:memory:"), KnowledgeBase(ROOT / "app" / "knowledge_th.csv", allow_unreviewed=True))


reset_agent()
print("พร้อมใช้โมเดล:", MODEL)

## หน้าแชต

พิมพ์แล้วกด Enter หรือ **ส่ง** · ลองเริ่มด้วย "น้ำท่วมบ้านค่ะ" → กดยินยอม → เล่าเหตุ เช่น "น้ำสูงถึงเอว ออกไม่ได้ มีผู้สูงอายุ 1 คน ต้องการเรือ" → ส่งตำแหน่ง → ตอบชื่อ/เบอร์ → กดยืนยัน

In [ ]:
import html
import uuid
import warnings

import ipywidgets as widgets
from IPython.display import display

from app.line_adapter import build_messages
from app.schemas import AFFECTED_LABELS_TH, now_bkk

LOCATIONS = {
    "อ.เมืองสิงห์บุรี": (14.8936, 100.3967),
    "อ.อินทร์บุรี": (14.9690, 100.3270),
    "อ.พรหมบุรี": (14.7830, 100.4360),
    "อ.บางระจัน": (14.8960, 100.3110),
    "กรุงเทพฯ (นอกพื้นที่)": (13.7563, 100.5018),
}
GREEN, BG = "#06c755", "#8cabd9"

ui_css = widgets.HTML("""<style>
.lc-phone { background:#8cabd9; border-radius:18px; overflow:hidden; box-shadow:0 6px 24px rgba(0,0,0,.25); }
.lc-bar { background:#fff; padding:6px 8px; border-top:1px solid #dcdfe4; }
.lc-quick { background:#8cabd9; padding:6px 8px 8px; overflow-x:auto; }
.lc-chip button, button.lc-chip { background:#fff !important; color:#06893a !important; border:1px solid #06c755 !important;
  border-radius:18px !important; font-size:13px !important; }
.lc-send button, button.lc-send { background:none !important; color:#06c755 !important; font-weight:700 !important; }
.lc-icon button, button.lc-icon { background:none !important; font-size:18px !important; }
.lc-input input { border-radius:18px !important; background:#eef0f3 !important; border:0 !important; padding:0 12px !important; color:#111 !important; }
.lc-loc { background:#fff; padding:8px 10px; border-top:1px solid #dcdfe4; }
</style>""")


def esc(text):
    return html.escape(text).replace("\n", "<br>")


def hhmm():
    return now_bkk().strftime("%H:%M")


# ---------- บทสนทนาแยกตามผู้ใช้ (สลับ user_id แล้วย้อนกลับมาคุยต่อได้) ----------
histories: dict[str, list[dict]] = {}
quick_items: dict[str, list] = {}
current_user = "line_" + uuid.uuid4().hex[:8]


def history():
    return histories.setdefault(current_user, [])


def bubble_html(m):
    meta = '<div style="font-size:10px;color:#44536b;white-space:nowrap;text-align:{a};margin:0 4px">{t}</div>'
    if m["who"] == "day":
        return (f'<div style="text-align:center;margin:4px 0"><span style="background:rgba(0,0,0,.18);color:#fff;'
                f'font-size:11px;padding:2px 10px;border-radius:10px">{esc(m["text"])}</span></div>')
    if m["who"] == "me":
        if m.get("loc"):
            lat, lng = m["loc"]
            body = ('<div style="width:200px;background:#fff;border-radius:14px;overflow:hidden;color:#111">'
                    '<div style="height:80px;display:flex;align-items:center;justify-content:center;font-size:28px;'
                    'background:repeating-linear-gradient(45deg,#cfe3c4 0 10px,#d9ebcf 10px 20px)">📍</div>'
                    f'<div style="padding:6px 10px;font-size:12px">ตำแหน่ง {lat:.5f}, {lng:.5f}</div></div>')
        else:
            body = (f'<div style="background:#8de055;color:#111;padding:8px 12px;border-radius:18px;border-top-right-radius:4px;'
                    f'max-width:260px;line-height:1.45;font-size:14px;word-break:break-word">{esc(m["text"])}</div>')
        read = "อ่านแล้ว<br>" if m.get("read") else ""
        return (f'<div style="display:flex;flex-direction:row-reverse;align-items:flex-end;gap:4px;margin:8px 0">{body}'
                + meta.format(a="right", t=read + m["time"]) + "</div>")
    avatar = ('<div style="width:32px;height:32px;border-radius:50%;background:#06c755;color:#fff;flex:none;display:flex;'
              'align-items:center;justify-content:center;font-size:16px">🌊</div>')
    if m["who"] == "typing":
        text = '<span style="letter-spacing:3px;color:#999">● ● ●</span>'
        bg, fg = "#fff", "#111"
    else:
        text = esc(m["text"])
        bg, fg = ("#fde7e7", "#c5221f") if m["who"] == "err" else ("#fff", "#111")
    body = (f'<div style="background:{bg};color:{fg};padding:8px 12px;border-radius:18px;border-top-left-radius:4px;'
            f'max-width:260px;line-height:1.45;font-size:14px;word-break:break-word">{text}</div>')
    return (f'<div style="display:flex;align-items:flex-end;gap:6px;margin:8px 0"><div style="align-self:flex-start">{avatar}</div>'
            + body + meta.format(a="left", t=m.get("time", "")) + "</div>")


chat_header = widgets.HTML()
chat_log = widgets.HTML(layout=widgets.Layout(width="100%"))
chat_quick = widgets.HBox(layout=widgets.Layout(flex_flow="row nowrap", overflow_x="auto", gap="6px"))
chat_quick.add_class("lc-quick")
chat_text = widgets.Text(placeholder="พิมพ์ข้อความ", layout=widgets.Layout(flex="1", height="34px"))
chat_text.add_class("lc-input")
chat_send = widgets.Button(description="ส่ง", layout=widgets.Layout(width="52px"))
chat_send.add_class("lc-send")
chat_pin = widgets.Button(description="📍", tooltip="ส่งตำแหน่ง", layout=widgets.Layout(width="40px"))
chat_pin.add_class("lc-icon")
loc_preset = widgets.Dropdown(options=list(LOCATIONS), description="ที่ตั้ง", layout=widgets.Layout(width="100%"))
loc_lat = widgets.FloatText(description="Lat", layout=widgets.Layout(width="49%"))
loc_lng = widgets.FloatText(description="Lng", layout=widgets.Layout(width="49%"))
loc_go = widgets.Button(description="ส่งตำแหน่ง", button_style="success")
loc_cancel = widgets.Button(description="ยกเลิก")
loc_box = widgets.VBox([loc_preset, widgets.HBox([loc_lat, loc_lng]), widgets.HBox([loc_cancel, loc_go])],
                       layout=widgets.Layout(display="none"))
loc_box.add_class("lc-loc")
input_bar = widgets.HBox([chat_pin, chat_text, chat_send], layout=widgets.Layout(align_items="center"))
input_bar.add_class("lc-bar")
phone = widgets.VBox([chat_header, chat_log, chat_quick, loc_box, input_bar], layout=widgets.Layout(width="400px"))
phone.add_class("lc-phone")

user_box = widgets.Text(description="user_id", layout=widgets.Layout(width="100%"))
new_user = widgets.Button(description="↺ ผู้ใช้ใหม่", tooltip="เริ่มคุยในฐานะผู้ใช้อีกคน (DB เดิม)")
wipe_db = widgets.Button(description="ล้าง DB ทั้งหมด", button_style="danger")
state_html = widgets.HTML()
panel = widgets.VBox([widgets.HTML("<b>ผู้ใช้ทดสอบ</b>"), user_box, widgets.HBox([new_user, wipe_db]), state_html],
                     layout=widgets.Layout(width="380px", padding="0 0 0 16px"))
chat_ui = widgets.VBox([ui_css, widgets.HBox([phone, panel], layout=widgets.Layout(flex_flow="row wrap"))])


def paint():
    chat_header.value = (
        '<div style="background:#2c3a4f;color:#fff;padding:10px 14px;display:flex;align-items:center;gap:10px">'
        '<div style="width:32px;height:32px;border-radius:50%;background:#06c755;display:flex;align-items:center;'
        'justify-content:center">🌊</div><div><b>ศูนย์รับแจ้งเหตุน้ำท่วม</b>'
        f'<div style="font-size:11px;opacity:.7">{esc(current_user)}</div></div></div>')
    inner = "".join(bubble_html(m) for m in history())
    # column-reverse ทำให้กล่องเลื่อนไปข้อความล่าสุดเองโดยไม่ต้องใช้ JavaScript
    chat_log.value = (f'<div style="height:480px;overflow-y:auto;display:flex;flex-direction:column-reverse;'
                      f'background:{BG};padding:8px 10px"><div>{inner}</div></div>')


def paint_quick():
    chips = []
    for q in quick_items.get(current_user, []):
        chip = widgets.Button(description=("📍 " if q.type == "location" else "") + q.label,
                              layout=widgets.Layout(width="auto", flex="none"))
        chip.add_class("lc-chip")
        chip.on_click(lambda _, q=q: open_location() if q.type == "location" else turn("postback", q.data, q.label))
        chips.append(chip)
    chat_quick.children = tuple(chips)
    chat_quick.layout.display = "flex" if chips else "none"


def row(label, value):
    return (f'<tr><td style="color:#888;padding:2px 10px 2px 0;vertical-align:top;white-space:nowrap">{label}</td>'
            f'<td style="word-break:break-all">{value}</td></tr>')


def paint_state(out=None):
    rows = []
    if out is not None:
        rows += [row("case_id", out.case_id or "—"),
                 row("intake_status", out.intake_status.value if out.intake_status else "—"),
                 row("pending_question", out.pending_question or "—"),
                 row("missing_slots", ", ".join(s.value for s in out.missing_slots) or "—"),
                 row("kb_doc_ids", ", ".join(out.kb_doc_ids) or "—")]
    case_id = agent.store.user(current_user).get("current_case_id")
    case = agent.store.load_case(case_id) if case_id else None
    if case is not None:
        inc = case.incident
        people = ", ".join(f"{AFFECTED_LABELS_TH[k]} {v}" for k, v in inc.affected_people.model_dump().items() if v)
        rows += [row("ปัญหา", esc(inc.problem or "—")), row("ต้องการ", esc(inc.needs or "—")),
                 row("ผู้ประสบภัย", people or "—"),
                 row("Priority", f"<b>{case.priority.level.value if case.priority.level else '—'}</b> · "
                                 f"{esc(case.priority.reason)} <span style='color:#888'>({case.priority_rule_id})</span>"),
                 row("Risk facts", esc(str({k: v for k, v in case.risk_facts.model_dump().items() if v != "unknown"}))),
                 row("Slots", "<br>".join(f"{k.value}: {v.status.value} (ถาม {v.ask_count})" for k, v in case.slots.items())),
                 row("Draft / Version", f"rev {case.draft_rev} / v{case.current_version}")]
    table = "".join(rows) or row("—", "ยังไม่มีเคส")
    state_html.value = f'<b>สถานะเคส</b><table style="font-size:12px;margin-top:6px">{table}</table>'


def busy(on):
    chat_send.disabled = chat_pin.disabled = chat_text.disabled = loc_go.disabled = on
    for chip in chat_quick.children:
        chip.disabled = on


def turn(kind, value, label):
    msgs = history()
    me = {"who": "me", "text": label, "time": hhmm(), "loc": value if kind == "location" else None}
    msgs += [me, {"who": "typing"}]
    quick_items[current_user] = []
    busy(True)
    paint(); paint_quick()
    try:
        out = agent.handle(current_user, **{kind: value})
        me["read"] = True
        msgs.pop()
        # build_messages ตัวเดียวกับที่ส่งเข้า LINE จริง: แบ่งข้อความยาว + ติดปุ่ม Quick Reply
        for m in build_messages(out):
            msgs.append({"who": "bot", "text": m["text"], "time": hhmm()})
        quick_items[current_user] = out.quick_replies
        paint_state(out)
    except Exception as exc:
        msgs.pop()
        msgs.append({"who": "err", "text": f"⚠️ {type(exc).__name__}: {str(exc)[:300]}", "time": hhmm()})
    finally:
        busy(False)
        paint(); paint_quick()


def on_send(_=None):
    text = chat_text.value.strip()
    if text:
        chat_text.value = ""
        turn("text", text, text)


def open_location(_=None):
    loc_preset.value = list(LOCATIONS)[0]
    loc_lat.value, loc_lng.value = LOCATIONS[loc_preset.value]
    loc_box.layout.display = "flex"


def close_location(_=None):
    loc_box.layout.display = "none"


def send_location(_):
    lat, lng = loc_lat.value, loc_lng.value
    if not (-90 <= lat <= 90 and -180 <= lng <= 180):
        loc_lat.description = "Lat ❌"
        return
    loc_lat.description = "Lat"
    close_location()
    turn("location", (lat, lng), "")


def switch_user(uid):
    global current_user
    current_user = uid
    user_box.value = uid
    if not histories.get(uid):
        histories[uid] = [{"who": "day", "text": now_bkk().strftime("วันนี้ %d/%m")},
                          {"who": "bot", "text": "(โหมดทดสอบ) พิมพ์ข้อความแรกเหมือนผู้ใช้เพิ่งทักแชต เช่น \"น้ำท่วมบ้านค่ะ\"",
                           "time": ""}]
    close_location()
    paint(); paint_quick(); paint_state()


def on_wipe(_):
    reset_agent()
    histories.clear(); quick_items.clear()
    switch_user("line_" + uuid.uuid4().hex[:8])


chat_send.on_click(on_send)
with warnings.catch_warnings():
    warnings.simplefilter("ignore", DeprecationWarning)
    chat_text.on_submit(on_send)  # Enter เพื่อส่ง
chat_pin.on_click(open_location)
loc_cancel.on_click(close_location)
loc_go.on_click(send_location)
loc_preset.observe(lambda c: setattr(loc_lat, "value", LOCATIONS[c["new"]][0]) or setattr(loc_lng, "value", LOCATIONS[c["new"]][1]),
                   "value")
new_user.on_click(lambda _: switch_user("line_" + uuid.uuid4().hex[:8]))
user_box.observe(lambda c: c["new"].strip() and c["new"].strip() != current_user and switch_user(c["new"].strip()), "value")
user_box.continuous_update = False
wipe_db.on_click(on_wipe)

switch_user(current_user)
display(chat_ui)

## ดูข้อมูลที่ Pipeline เห็น (เฉพาะเคสที่ยืนยันแล้ว)

In [ ]:
agent.store.pipeline_rows()

## ดูข้อมูลที่บันทึกลง Database จริง

รันเซลล์นี้หลังสร้าง Agent แล้ว กด **โหลดข้อมูลที่บันทึก** หลังคุยแต่ละรอบ เลือก user_id ให้ตรงกับหน้าจอแชต

เปิด `case_personal_data` เพื่อดูชื่อ นามสกุล เบอร์ และพิกัด; `case_drafts` เพื่อดูร่าง; หลังยืนยันดู `case_versions` และ `case_version_contacts` รวมถึง View ของ Pipeline

ส่วนนี้อ่านแถวจากฐานข้อมูลโดยตรง ไม่ใช่ข้อความตอบของ LLM โหมด Mock ไม่สร้างแถวจริง SQLite ในหน่วยความจำจะหายเมื่อ reset หรือปิด kernel และข้อมูลส่วนบุคคลอาจติดใน output เมื่อบันทึก notebook จึงควรใช้ข้อมูลสมมติ


In [ ]:
import json
import ipywidgets as widgets
from IPython.display import display

# อ่านจาก connection ของ Agent โดยตรง ไม่ต้องใช้ admin token หรือ HTTP
saved_user = widgets.Text(
    value=globals().get("current_user", globals().get("USER", "lab_user")),
    description="ผู้แจ้ง:", continuous_update=False,
)
saved_refresh = widgets.Button(description="โหลดข้อมูลที่บันทึก", button_style="info")
saved_status = widgets.Label()
saved_tables = widgets.Accordion(children=[])


def refresh_saved_database(_=None):
    if globals().get("agent") is None:
        saved_tables.children = ()
        saved_status.value = "รันเซลล์ตั้งค่าและสร้าง Agent ก่อน (Mock ไม่บันทึกข้อมูล)"
        return
    user_ref = saved_user.value.strip()
    database = agent.store
    result = {}
    with database.transaction() as cursor:
        queries = {
            "users": "SELECT * FROM users WHERE user_ref=?",
            "cases": "SELECT * FROM cases WHERE user_ref=?",
            "case_raw_messages": "SELECT * FROM case_raw_messages WHERE user_ref=? ORDER BY created_at",
        }
        for table in ("case_personal_data", "case_drafts", "case_slots", "case_versions",
                      "case_version_contacts", "case_events", "pipeline_confirmed_cases", "pipeline_case_contacts"):
            queries[table] = f"SELECT * FROM {table} WHERE case_id IN (SELECT case_id FROM cases WHERE user_ref=?)"
        for table, sql in queries.items():
            rows = database._exec(cursor, sql, (user_ref,))
            columns = [column[0] for column in rows.description]
            result[table] = [dict(zip(columns, row)) for row in rows.fetchall()]
    panels = []
    for table, rows in result.items():
        panels.append(widgets.Textarea(
            value=json.dumps(rows, ensure_ascii=False, indent=2, default=str), disabled=True,
            layout=widgets.Layout(width="100%", height="260px"),
        ))
    saved_tables.children = tuple(panels)
    for index, (table, rows) in enumerate(result.items()):
        saved_tables.set_title(index, f"{table} — {len(rows)} แถว")
    saved_status.value = ("PostgreSQL" if database.postgres else "SQLite") + " · ข้อมูลจริงของ " + user_ref


saved_refresh.on_click(refresh_saved_database)
saved_user.observe(refresh_saved_database, names="value")
display(widgets.VBox([widgets.HBox([saved_user, saved_refresh]), saved_status, saved_tables]))
refresh_saved_database()
